# 02 — ETL: Limpeza e Transformação

**Objetivo:** Consolidar todos os CSVs semestrais da ANP, limpar, padronizar e enriquecer com dados de câmbio e petróleo.

**Ferramentas:** Pandas + DuckDB

**Etapas:**
1. Consolidação dos CSVs semestrais
2. Padronização de colunas
3. Limpeza (nulos, formatos, duplicatas)
4. Enriquecimento com dólar e Brent
5. Agregações úteis
6. Queries analíticas com DuckDB
7. Exportação em Parquet

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import duckdb

from src.etl import (
    carregar_csvs_anp, padronizar_colunas, limpar_dados,
    enriquecer_com_dados_externos, criar_aggregacoes,
    executar_queries_duckdb, executar_pipeline_etl
)
from src.utils import DATA_RAW, DATA_PROCESSED

## 2.1 — Consolidação dos CSVs

In [2]:
csvs_disponiveis = sorted(DATA_RAW.glob("*.csv"))
print(f"Total de arquivos CSV disponíveis: {len(csvs_disponiveis)}")
print(f"Exemplos: {[f.name for f in csvs_disponiveis[:5]]}")

df_raw = pd.concat(
    [pd.read_csv(f, sep=';', encoding='latin-1', low_memory=False) for f in csvs_disponiveis[:2]],
    ignore_index=True,
)
print(f"\nShape (amostra 2 arquivos): {df_raw.shape}")
print(f"Colunas: {list(df_raw.columns)}")
print(f"Memória: {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df_raw.head()

Total de arquivos CSV disponíveis: 44
Exemplos: ['1º_semestre_de_2004.csv', '1º_semestre_de_2005.csv', '1º_semestre_de_2006.csv', '1º_semestre_de_2007.csv', '1º_semestre_de_2008.csv']



Shape (amostra 2 arquivos): (1194260, 16)
Colunas: ['ï»¿Regiao - Sigla', 'Estado - Sigla', 'Municipio', 'Revenda', 'CNPJ da Revenda', 'Nome da Rua', 'Numero Rua', 'Complemento', 'Bairro', 'Cep', 'Produto', 'Data da Coleta', 'Valor de Venda', 'Valor de Compra', 'Unidade de Medida', 'Bandeira']
Memória: 1108.9 MB


,ï»¿Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
0,SE,SP,GUARULHOS,AUTO POSTO SAKAMOTO LTDA,49.051.667/0001-02,RODOVIA PRESIDENTE DUTRA,S/N,"KM 210,5-SENT SP/RJ",BONSUCESSO,07178-580,GASOLINA,11/05/2004,"1,967","1,6623",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
1,SE,SP,GUARULHOS,AUTO POSTO SAKAMOTO LTDA,49.051.667/0001-02,RODOVIA PRESIDENTE DUTRA,S/N,"KM 210,5-SENT SP/RJ",BONSUCESSO,07178-580,ETANOL,11/05/2004,"0,899","0,6282",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
2,SE,SP,GUARULHOS,AUTO POSTO SAKAMOTO LTDA,49.051.667/0001-02,RODOVIA PRESIDENTE DUTRA,S/N,"KM 210,5-SENT SP/RJ",BONSUCESSO,07178-580,DIESEL,11/05/2004,"1,299","1,1704",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
3,SE,SP,SOROCABA,COMPETRO COMERCIO E DISTRIBUICAO DE DERIVADOS ...,00.003.188/0001-21,RUA HUMBERTO DE CAMPOS,306,NaN,JARDIM ZULMIRA,18061-000,GASOLINA,10/05/2004,"1,85","1,67",R$ / litro,BRANCA
4,SE,SP,SOROCABA,COMPETRO COMERCIO E DISTRIBUICAO DE DERIVADOS ...,00.003.188/0001-21,RUA HUMBERTO DE CAMPOS,306,NaN,JARDIM ZULMIRA,18061-000,ETANOL,10/05/2004,"0,78","0,48",R$ / litro,BRANCA


## 2.2 — Padronização de colunas

In [3]:
df = padronizar_colunas(df_raw)
print("Colunas padronizadas:")
print(list(df.columns))
df.head()

Colunas padronizadas:
['i_regiao_sigla', 'estado', 'municipio', 'revenda', 'cnpj', 'rua', 'numero', 'complemento', 'bairro', 'cep', 'produto', 'data_coleta', 'valor_venda', 'valor_compra', 'unidade', 'bandeira']


,i_regiao_sigla,estado,municipio,revenda,cnpj,rua,numero,complemento,bairro,cep,produto,data_coleta,valor_venda,valor_compra,unidade,bandeira
0,SE,SP,GUARULHOS,AUTO POSTO SAKAMOTO LTDA,49.051.667/0001-02,RODOVIA PRESIDENTE DUTRA,S/N,"KM 210,5-SENT SP/RJ",BONSUCESSO,07178-580,GASOLINA,11/05/2004,"1,967","1,6623",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
1,SE,SP,GUARULHOS,AUTO POSTO SAKAMOTO LTDA,49.051.667/0001-02,RODOVIA PRESIDENTE DUTRA,S/N,"KM 210,5-SENT SP/RJ",BONSUCESSO,07178-580,ETANOL,11/05/2004,"0,899","0,6282",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
2,SE,SP,GUARULHOS,AUTO POSTO SAKAMOTO LTDA,49.051.667/0001-02,RODOVIA PRESIDENTE DUTRA,S/N,"KM 210,5-SENT SP/RJ",BONSUCESSO,07178-580,DIESEL,11/05/2004,"1,299","1,1704",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
3,SE,SP,SOROCABA,COMPETRO COMERCIO E DISTRIBUICAO DE DERIVADOS ...,00.003.188/0001-21,RUA HUMBERTO DE CAMPOS,306,NaN,JARDIM ZULMIRA,18061-000,GASOLINA,10/05/2004,"1,85","1,67",R$ / litro,BRANCA
4,SE,SP,SOROCABA,COMPETRO COMERCIO E DISTRIBUICAO DE DERIVADOS ...,00.003.188/0001-21,RUA HUMBERTO DE CAMPOS,306,NaN,JARDIM ZULMIRA,18061-000,ETANOL,10/05/2004,"0,78","0,48",R$ / litro,BRANCA


## 2.3 — Limpeza dos dados

In [4]:
print("ANTES DA LIMPEZA")
print(f"Shape: {df.shape}")
print(f"\nNulos por coluna:")
print(df.isnull().sum())
print(f"\nTipos:")
print(df.dtypes)

ANTES DA LIMPEZA
Shape: (1194260, 16)

Nulos por coluna:
i_regiao_sigla         0
estado                 0
municipio              0
revenda                0
cnpj                   0
rua                    0
numero               677
complemento       804786
bairro              3435
cep                    0
produto                0
data_coleta            0
valor_venda            0
valor_compra      386336
unidade                0
bandeira               0
dtype: int64

Tipos:
i_regiao_sigla    object
estado            object
municipio         object
revenda           object
cnpj              object
rua               object
numero            object
complemento       object
bairro            object
cep               object
produto           object
data_coleta       object
valor_venda       object
valor_compra      object
unidade           object
bandeira          object
dtype: object


In [5]:
df = limpar_dados(df)
print(f"\nAPÓS LIMPEZA")
print(f"Shape: {df.shape}")
print(f"\nProdutos padronizados: {df['produto_padronizado'].unique()}")
print(f"Estados: {sorted(df['estado_sigla'].unique())}")
print(f"Período: {df['data_coleta'].min()} a {df['data_coleta'].max()}")
print(f"\nEstatísticas de preço:")
df['valor_venda'].describe()


APÓS LIMPEZA
Shape: (1194116, 21)

Produtos padronizados: ['Gasolina Comum' 'Etanol' 'Diesel' 'GNV']
Estados: ['AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RR', 'RS', 'SC', 'SE', 'SP', 'TO']
Período: 2004-05-10 00:00:00 a 2005-06-30 00:00:00

Estatísticas de preço:


count    1.194116e+06
mean     1.751437e+00
std      4.392104e-01
min      5.900000e-01
25%      1.459000e+00
50%      1.690000e+00
75%      2.149000e+00
max      3.220000e+00
Name: valor_venda, dtype: float64

## 2.4 — Enriquecimento com dados externos

In [6]:
df = enriquecer_com_dados_externos(df)
print("Colunas após enriquecimento:")
print(list(df.columns))

if 'dolar_venda' in df.columns:
    print(f"\nDólar — média: R$ {df['dolar_venda'].mean():.2f}")
if 'preco_brent_usd' in df.columns:
    print(f"Brent — média: US$ {df['preco_brent_usd'].mean():.2f}")

Colunas após enriquecimento:
['i_regiao_sigla', 'estado', 'municipio', 'revenda', 'cnpj', 'rua', 'numero', 'complemento', 'bairro', 'cep', 'produto', 'data_coleta', 'valor_venda', 'valor_compra', 'unidade', 'bandeira', 'produto_padronizado', 'estado_sigla', 'ano', 'mes', 'ano_mes', 'dolar_venda', 'preco_brent_usd']

Dólar — média: R$ 2.71
Brent — média: US$ 46.52


## 2.5 — Salvando dados processados

In [7]:
path_parquet = DATA_PROCESSED / 'combustiveis_brasil.parquet'

if path_parquet.exists():
    print(f"✅ Parquet já existe: {path_parquet.name}")
    print(f"   Tamanho: {path_parquet.stat().st_size / 1e6:.1f} MB")
    with duckdb.connect() as con:
        r = con.execute(f"""
            SELECT COUNT(*) as n, MIN(ano) as inicio, MAX(ano) as fim
            FROM read_parquet('{path_parquet}')
        """).fetchdf().iloc[0]
    print(f"   Registros: {int(r['n']):,}  |  Período: {int(r['inicio'])}–{int(r['fim'])}")
    for f in sorted(DATA_PROCESSED.glob("agg_*.parquet")):
        print(f"   {f.name}: {f.stat().st_size / 1e6:.1f} MB")
else:
    print("Executando pipeline ETL completo (pode levar vários minutos)...")
    executar_pipeline_etl(salvar_parquet=True, tamanho_lote=10)
    print("✅ Pipeline concluído!")

✅ Parquet já existe: combustiveis_brasil.parquet
   Tamanho: 440.3 MB
   Registros: 20,484,968  |  Período: 2004–2025
   agg_mensal_estado_produto.parquet: 0.5 MB
   agg_mensal_nacional.parquet: 0.0 MB
   agg_por_bandeira.parquet: 0.7 MB
   agg_presidente_prudente.parquet: 0.0 MB


## 2.6 — Queries analíticas com DuckDB

Demonstração de SQL sobre arquivos Parquet sem necessidade de banco de dados.

In [8]:
resultados = executar_queries_duckdb(path_parquet)

print("\n=== Top 10 Estados Mais Caros (Gasolina Comum) ===")
display(resultados['top_estados_gasolina'])

print("\n=== Evolução Anual do Preço Médio ===")
display(resultados['evolucao_anual'].head(15))

print("\n=== Presidente Prudente vs SP vs Brasil ===")
display(resultados['pp_vs_sp_vs_brasil'].tail(12))


=== Top 10 Estados Mais Caros (Gasolina Comum) ===


,estado_sigla,preco_medio,n_registros
0,AC,3.857187,36535
1,TO,3.668274,46299
2,RO,3.645603,72911
3,MT,3.644872,121636
4,PA,3.593936,138253
5,CE,3.552620,194346
6,GO,3.482167,215323
7,MA,3.471513,116776
8,BA,3.470019,324378
9,AM,3.467848,90683



=== Evolução Anual do Preço Médio ===


,ano,produto_padronizado,preco_medio,preco_mediano,n_registros
0,2004,Diesel,1.516346,1.499,359036
1,2004,Etanol,1.287938,1.290,391982
2,2004,GNV,1.091518,1.099,21165
3,2004,Gasolina Comum,2.142971,2.139,424665
4,2005,Diesel,1.741082,1.700,493405
5,2005,Etanol,1.483699,1.500,543425
6,2005,GNV,1.152589,1.139,32312
7,2005,Gasolina Comum,2.358596,2.349,585256
8,2006,Diesel,1.880986,1.870,305058
9,2006,Etanol,1.701528,1.740,362463



=== Presidente Prudente vs SP vs Brasil ===


,ano,mes,produto_padronizado,preco_pp,preco_sp,preco_brasil
208,2025,1,Gasolina Comum,6.103788,6.017835,6.183839
209,2025,2,Gasolina Comum,6.273036,6.184026,6.367177
210,2025,3,Gasolina Comum,6.172143,6.176348,6.349952
211,2025,4,Gasolina Comum,6.116071,6.150155,6.316235
212,2025,5,Gasolina Comum,6.101786,6.138968,6.283602
213,2025,6,Gasolina Comum,6.013714,6.075125,6.226324
214,2025,7,Gasolina Comum,5.965893,6.056819,6.211219
215,2025,8,Gasolina Comum,5.966491,6.043537,6.182066
216,2025,9,Gasolina Comum,6.003750,6.064101,6.191606
217,2025,10,Gasolina Comum,6.015357,6.057356,6.215174


In [9]:
with duckdb.connect() as con:
    resultado_custom = con.execute(f"""
        SELECT 
            ano,
            produto_padronizado,
            ROUND(AVG(valor_venda), 2) as preco_medio,
            ROUND(STDDEV(valor_venda), 2) as desvio_padrao,
            COUNT(*) as registros
        FROM read_parquet('{path_parquet}')
        WHERE produto_padronizado IN ('Gasolina Comum', 'Etanol', 'Diesel')
        GROUP BY ano, produto_padronizado
        ORDER BY ano DESC, produto_padronizado
        LIMIT 15
    """).fetchdf()

display(resultado_custom)

,ano,produto_padronizado,preco_medio,desvio_padrao,registros
0,2025,Diesel,6.15,0.40,85969
1,2025,Etanol,4.46,0.46,177753
2,2025,Gasolina Comum,6.25,0.41,212423
3,2024,Diesel,5.97,0.37,102581
4,2024,Etanol,4.07,0.51,197306
5,2024,Gasolina Comum,5.93,0.42,233014
6,2023,Diesel,5.76,0.59,106136
7,2023,Etanol,4.00,0.52,197849
8,2023,Gasolina Comum,5.53,0.44,232085
9,2022,Diesel,6.63,0.72,102477


---
**Próximo passo:** [03_eda.ipynb](03_eda.ipynb) — Análise Exploratória de Dados